# Save tagged-unit figures

Independent notebook that loads one opto-tagging session, selects the **tagged units**
(significant, QC-passed, for a chosen condition/probe/shank), and saves a **separate**
raster + PSTH figure for **each** unit. Each figure includes the unit's mean waveform
as an inset in the top-right corner (`show_waveform=True`).

This notebook only reads the previously computed significance CSV, so it does not
recompute `find_significant_units`.

In [ ]:
# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")

In [ ]:
# =============================================================================
# 2. LOAD SESSION
# =============================================================================
import glob
from optical_tagging import OpticalTagging

session_date = '863973_2026-08-25_14-18-29'
behavior_json_file = f'/root/capsule/data/ecephys_{session_date}/behavior/{session_date}.json'

# The sorted folder has an extra timestamp not contained in session_date,
# so locate it by glob (one match expected per session).
ephys_nwb_file = glob.glob(
    f'/root/capsule/data/ecephys_{session_date}_sorted_*/nwb/ecephys_{session_date}_experiment1_recording1.nwb'
)[0]

OptoTag = OpticalTagging(
    behavior_json_file=behavior_json_file,
    ephys_nwb_file=ephys_nwb_file,
)

sig_csv_path = f'/root/capsule/scratch/opto_tagging/sig_{session_date}.csv'
print(f"OptoTag ready. Significance CSV: {sig_csv_path}")

In [ ]:
# =============================================================================
# 3. SELECT TAGGED UNITS (significant + QC-passed) FOR ONE CONDITION
#    Includes ALL probes and ALL shanks, and does NOT differentiate laser_name
#    (the laser name is still shown in each plot's title via the condition).
# =============================================================================
import types
import pandas as pd

# ---- Choose the condition to plot (all probes / all shanks are included) ----
# The condition is the 6-tuple used by find_significant_units:
#   (target_power, location_tag, laser_name, cycle_duration, frequency, pulse_duration)
#     5.0       -> target_power     : commanded laser power (mW)
#     1.0       -> location_tag      : stimulation location / site id
#     "Laser_1" -> laser_name        : which laser was used (IGNORED for selection)
#     1.0       -> cycle_duration    : laser-on duration of each train/cycle (s)
#     10.0      -> frequency         : pulse rate within the train (Hz) -> 10 pulses/s
#     0.05      -> pulse_duration    : duration of a single pulse (s) -> 50 ms
target_cond = (5.0, 1.0, "Laser_1", 1.0, 10.0, 0.05)

# Load the significance CSV and parse the 'condition' column, which is stored
# as strings like "(np.float64(5.0), ...)".
df = pd.read_csv(sig_csv_path)
fake_np = types.SimpleNamespace(float64=float)

def safe_parse_condition(x):
    if isinstance(x, str):
        try:
            return eval(x, {"np": fake_np})
        except Exception as e:
            print(f"Failed to parse: {x}\nError: {e}")
            return None
    return x

df['condition'] = df['condition'].apply(safe_parse_condition)

# Match the condition on every field EXCEPT laser_name (index 2), so units tagged
# by any laser under the same power/location/timing are all included.
IGNORE_IDX = 2  # laser_name position in the 6-tuple
match_idx = [i for i in range(6) if i != IGNORE_IDX]

def matches_ignore_laser(c):
    if not isinstance(c, (tuple, list)) or len(c) != 6:
        return False
    return all(c[i] == target_cond[i] for i in match_idx)

cond_mask = df['condition'].apply(matches_ignore_laser)

# Significant, positively-modulated units for this condition (any probe / shank / laser).
sigrows = df[
    cond_mask &
    (df["significant"] == True) &
    (df["stim_FR_mean"] > df["baseline_FR_mean"])
]

# Intersect with QC-passed units on the OptoTag object.
common_units = sorted(set(OptoTag.units_passing_qc.index) & set(sigrows['unit_id']))
print(f"Condition (laser_name ignored): {target_cond} | all probes / shanks / lasers")
print(f"Tagged units to plot ({len(common_units)}): {common_units}")


In [ ]:
# =============================================================================
# 4. SAVE ONE FIGURE PER TAGGED UNIT (raster + PSTH + waveform inset)
# =============================================================================
import os

# Output folder for the per-unit figures (all probes / shanks for this condition).
tagged_fig_dir = f"/root/capsule/results/tagged_units/{session_date}/all_probes_shanks/"
os.makedirs(tagged_fig_dir, exist_ok=True)

# plot_raster_graph supports 'pdf' and 'eps' (add 'png' if your version handles it).
SAVE_FORMATS = ["png", "eps"]

print(f"Saving {len(common_units)} tagged-unit figures to: {tagged_fig_dir}")
for u in common_units:
    # Per-unit path prefix -> plot_raster_graph appends the condition + extension,
    # so every unit (and condition) lands in its own file.
    unit_save_path = os.path.join(tagged_fig_dir, f"unit_{u}")
    OptoTag.plot_raster_graph(
        unit_index=u,
        bin_size=0.002,
        time_window=[-0.02, 0.09],
        removal_window=0.002,
        align_to_event="pulse",
        remove_artefacts=True,
        save_path=unit_save_path,
        save_formats=SAVE_FORMATS,
        show_waveform=True,   # mean waveform in the top-right corner
    )
    print(f"  saved unit {u}")

print("Done saving tagged-unit figures.")